In [1]:
from typing import Annotated,Literal
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from langchain_experimental.utilities import PythonREPL
from typing_extensions import TypedDict
from langgraph.graph import MessagesState,END,StateGraph,START
from langgraph.types import Command
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_groq import ChatGroq

llm=ChatGroq(model="Gemma2-9b-It")
llm.invoke("hello groq!").content

'Hello! 👋 \n\nHow can I help you today? 😊\n'

In [4]:
tavily_tool=TavilySearch()

In [5]:
tavily_tool.invoke('what is gdp?')

{'query': 'what is gdp?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'title': 'What Is GDP and Why Is It So Important to Economists and Investors?',
   'url': 'https://www.investopedia.com/ask/answers/what-is-gdp-why-its-important-to-economists-investors/',
   'content': 'GDP measures the total output of a national economy in a given period, adjusted for inflation. It is used by economists and investors to gauge economic performance, growth, and health.',
   'score': 0.8018276,
   'raw_content': None},
  {'title': 'Gross Domestic Product (GDP) Formula and How to Use It',
   'url': 'https://www.investopedia.com/terms/g/gdp.asp',
   'content': 'The GDP growth rate compares the annual or quarterly change in a country’s economic output to measure how fast an economy is growing. Nominal GDP is an assessment of economic production in an economy that includes current prices in its calculation. The GDP growth rate compares the year-over-year (or quarterly) chan

In [6]:
code="""
x=5
y=x*2
print(y)
"""

In [7]:
repl=PythonREPL()

In [8]:
repl.run(code)

Python REPL can execute arbitrary code. Use with caution.


'10\n'

In [9]:
@tool
def python_repl_tool(
    code:Annotated[str,"The python code to execute to generate your chart."],

):
    """Use this to execute python code and do math.If you want to see the output of a value, you should
    print it out with  `print(...)`. This is visible to the user."""
    try:
        result=repl.run(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"

    return f"Successfully executed: {code} Stdout: {result}"    

In [10]:

print(python_repl_tool.invoke(code))


Successfully executed: 
x=5
y=x*2
print(y)
 Stdout: 10



In [11]:
members=["researcher","coder"]

In [12]:
options=members+["FINISH"]

In [13]:
class Router(TypedDict):
    """ Worker to route to next. If no workers needed, route to FINISH. """
    next:Literal["researcher","coder","FINISH"]

In [14]:
class State(MessagesState):
    next:str

In [15]:
system_prompt=f"""
You are a supervisor tasked with managing a conversation between the following workers:{members}.
Given the following user request, respond with the worker to act next.
Each worker will perform a task and respond with their results and status.
When finished, respond with FINISH. 
"""

In [16]:
def supervisor_node(state: State):
    messages = [{'role': "system", "content": system_prompt}] + state["messages"]
    response = llm.with_structured_output(Router).invoke(messages)
    goto = response["next"]

    if goto == "FINISH":
        return Command(goto=END, update={"next": "FINISH"})

    return Command(goto=goto, update={"next": goto})


In [17]:
def research_node(state:State):
    research_agent=create_react_agent(llm,tools=[tavily_tool],prompt="You are a researcher. DO NOT do any math.")
    result=research_agent.invoke(state)
    return Command(
        update={
            "messages":[
                HumanMessage(content=result['messages'][-1].content,name="researcher")
            ]
        },
        goto="supervisor"
    )

In [18]:
def code_node(state:State):
    code_agent=create_react_agent(llm,tools=[python_repl_tool])
    result=code_agent.invoke(state)
    return Command(
        update={
            "messages":[
                HumanMessage(content=result["messages"][-1].content,name="coder")
            ]
        },
        goto="supervisor"
    )

In [ ]:
graph=StateGraph(State)
graph.add_node("supervisor",supervisor_node)
graph.add_node("researcher",research_node)
graph.add_node("coder",code_node)


graph.add_edge(START,"supervisor")
graph.add_conditional_edges(
    "supervisor",
    {
        "researcher": "researcher",
        "coder": "coder",
        END: END
    }
)
graph.add_edge("researcher", "supervisor")
graph.add_edge("coder", "supervisor")
app=graph.compile()

TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class 'str'>

In [ ]:
def fa():
    